# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">10/7(수) 오후 · 총평 · 보고서 틀 · 조건부 경고 — 실습</mark>

오전에는 경보 18건을 묶음으로 나눠 요약하고 위험한 것부터 줄 세웠습니다. 오후에는 그 결과로 **팀장이 아침에 읽을 보고서 한 장**을 만듭니다.

완성하면 이런 파일이 생깁니다.

```
# 야간 보안 관제 보고 (2026-10-07)

## 한눈에 보기
- 처리한 경보: 18건 (high 6건)
- **주의: high 경보 6건 — 건별 내역을 먼저 확인할 것**

## 총평
밤사이 관리자 계정을 노린 무차별 대입과 로그인 성공이 이어져 …

## 건별 내역 (위험한 것부터)
- [HIGH] E01 admin 계정에 로그인 실패 4회
…
```

**숫자와 틀은 코드가, 총평 문장은 LLM 이** 씁니다. 왜 그렇게 나누는지가 오늘 오후의 핵심입니다.

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기</mark>

### 0.1 맨 먼저

1. 이 파일을 오전 노트북과 같은 **새** `security-agent-toolkit/agent_core` 폴더에 넣습니다.
2. 오른쪽 위 **커널 선택**에서 오전과 같은 **각자 만든 가상환경**을 고릅니다.
3. 아래 **위치 확인 셀**과 **준비 셀 셋**을 차례로 실행합니다. 오전을 못 끝냈어도 여기서부터 할 수 있습니다.

### ⚠ 중요 · 무료 키는 1분에 15번까지만 호출할 수 있습니다

<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px; border-left:6px solid #e65100">
1. 한도를 넘으면 답을 받지 못합니다. 1분 기다린 뒤 같은 셀을 다시 실행합니다.<br>
2. 오후에 LLM 을 부르는 문제는 <b>5-3 · 5-5 · ⭐5-6 · 7-3 · 7-4</b> 입니다. 7-4 는 한 번 실행에 두 번 부릅니다.<br>
3. 한도에 걸리면 <code>[LLM 호출 실패] 상태 코드 429</code> 가 먼저 보이고, 그 아래에 <code>KeyError: 'choices'</code> 나 <code>TypeError: 'NoneType' object is not subscriptable</code> 같은 에러가 이어질 수 있습니다. <b>코드가 틀린 것이 아닙니다.</b> 1분 기다린 뒤 같은 셀을 다시 실행합니다.<br>
4. LLM 의 답 문장과 위험도는 실행할 때마다 달라질 수 있습니다. 🎯 와 글자가 같지 않아도 <b>모양이 같으면</b> 맞습니다.
</div>

### 0.2 오늘 오후의 순서

| 교시 | 무엇 | LLM |
|---|---|---|
| 5교시 | 요약을 다시 요약하기 — 총평 | 부름 |
| 6교시 | 보고서 틀 · 숫자는 코드가 · 조건부 경고 | 부르지 않음 |
| 7교시 | `report_generator.py` · 처음부터 끝까지 잇기 · 다른 날로 다시 | 부름 |
| 8교시 | 오늘 만든 것 확인 | — |

### 0.3 문제를 푸는 법

1. 문제 앞의 **🐍 문법 상자**를 먼저 읽습니다. 그 문제에서 쓰는 문법이 있습니다.
2. 코드 셀의 번호 주석이 무엇을 쓸지 알려 줍니다. 주석에 적힌 **변수 이름을 그대로** 씁니다.
3. 막히면 **💡 힌트**를 보고, 그래도 막히면 노트북 맨 아래 「정답」을 봅니다. 도전 문제는 안 풀고 넘어가도 됩니다.
4. `NameError` 가 나오면 위쪽 셀을 건너뛴 것입니다.

In [2]:
# 위치 확인 셀 — 그대로 실행합니다
import os                                                     # 폴더 · 파일을 다루는 도구
import sys                                                    # 지금 실행 중인 파이썬의 정보를 주는 도구


def find_env():                                               # .env 를 지금 폴더부터 위쪽으로 찾는 함수
    folder = os.getcwd()                                      # 지금 폴더에서 시작한다
    for i in range(5):                                        # 다섯 칸 위까지 찾는다
        path = os.path.join(folder, ".env")                   # 그 폴더 안의 .env 경로를 만든다
        if os.path.exists(path):                              # 그 파일이 있으면
            return path                                       # 찾은 경로를 돌려준다
        folder = os.path.dirname(folder)                      # 한 칸 위 폴더로 올라간다
    return None                                               # 못 읽으면 None


ENV_PATH = find_env()                                         # 찾은 .env 경로. 못 찾으면 None

print("노트북이 실행되는 폴더:", os.getcwd())
print("지금 쓰는 파이썬     :", sys.executable)

if ENV_PATH:
    print("키 파일(.env)         :", ENV_PATH)
    print("준비 완료")
else:                                                         # 아니면
    print(".env 를 찾지 못했습니다")
    print("1) 이 노트북을 새 security-agent-toolkit 폴더 안의 agent_core 폴더로 옮긴 뒤 다시 엽니다")
    print("2) 아침 과제 6 의 「.env 옮기기」를 했는지 봅니다")

노트북이 실행되는 폴더: c:\Users\User\Documents\security-agent-toolkit
지금 쓰는 파이썬     : c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\python.exe
키 파일(.env)         : c:\Users\User\Documents\security-agent-toolkit\.env
준비 완료


아래 세 셀도 실행합니다. `llm_client.py` 와 `event_summarizer.py` 를 오전 정답과 같은 내용으로 다시 만들고(이미 있으면 같은 내용으로 덮어씁니다), 오후에 쓸 데이터를 저장합니다.

In [3]:
%%writefile llm_client.py
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import os                                                     # 폴더 · 파일을 다루는 도구
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)

# Gemini 에 요청을 보낼 주소
URL = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
MODEL = "gemini-3.5-flash-lite"                               # 쓸 모델 이름


def find_env():                                               # .env 를 지금 폴더부터 위쪽으로 찾는 함수
    folder = os.getcwd()                                      # 지금 폴더에서 시작한다
    for i in range(5):                          # 지금 폴더에서 다섯 칸 위까지 찾습니다
        path = os.path.join(folder, ".env")                   # 그 폴더 안의 .env 경로를 만든다
        if os.path.exists(path):                              # 그 파일이 있으면
            return path                                       # 찾은 경로를 돌려준다
        folder = os.path.dirname(folder)        # 한 칸 위 폴더
    return None                                               # 못 읽으면 None


def read_api_key():                                           # .env 에서 키를 읽어 돌려주는 함수
    api_key = None                                            # 키를 못 찾으면 None 으로 남는다
    with open(find_env(), encoding="utf-8") as f:             # .env 를 찾아 읽기로 연다
        for line in f:                                        # 파일을 한 줄씩 꺼낸다
            parts = line.strip().split("=", 1)                # 줄 끝 공백을 지우고 = 에서 둘로 나눈다
            if parts[0] == "GEMINI_API_KEY":                  # = 앞부분이 키 이름이면
                api_key = parts[1]                            # = 뒷부분이 키 값이다
    return api_key                                            # 읽은 키를 돌려준다


def call_llm(question):                                       # 질문을 받아 답 문장을 돌려주는 함수
    # 요청 헤더 — .env 에서 읽은 키를 넣는다
    headers = {"Authorization": "Bearer " + read_api_key(), "Content-Type": "application/json"}
    # 요청 본문 — 모델 · temperature 0 · 질문
    body = {"model": MODEL, "temperature": 0, "messages": [{"role": "user", "content": question}]}
    # POST 요청을 보낸다. 30초 안에 답이 없으면 멈춘다
    response = requests.post(URL, headers=headers, json=body, timeout=30)
    if response.status_code != 200:                           # 200(성공)이 아니면
        print("[LLM 호출 실패] 상태 코드", response.status_code)      # 429 면 1분 기다린 뒤 다시 실행
        return ""                                             # 실패하면 빈 문자열을 돌려준다
    data = response.json()                                    # 응답(JSON 문자열)을 딕셔너리로 바꾼다
    return data["choices"][0]["message"]["content"]           # 답 문장만 돌려준다


def parse_llm_json(text):                                     # LLM 의 답을 딕셔너리로 바꾸는 함수. 못 읽으면 None
    clean = text.replace("```json", "")                       # 코드 블록 표시 ```json 을 지운다
    clean = clean.replace("```", "")                          # 남은 ``` 를 지운다
    clean = clean.strip()                                     # 앞뒤 공백 · 줄바꿈을 지운다

    try:                                                      # 아래 줄을 해 본다
        return json.loads(clean)                              # 문자열을 딕셔너리로 바꿔 돌려준다
    except json.JSONDecodeError:                              # JSON 으로 읽지 못하면 여기로 온다
        return None                                           # 못 읽으면 None

Overwriting llm_client.py


In [4]:
%%writefile event_summarizer.py
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
from llm_client import call_llm, parse_llm_json               # llm_client.py 의 두 함수를 꺼내 쓴다

SIZE = 6                                                      # 한 묶음에 넣을 경보 수
# 요약 지시 — 출력 형식(JSON 배열)과 키 이름을 정한다
instruction = "다음 보안 경보 목록을 읽고, 경보마다 id, risk_level, summary 세 키를 가진 JSON 배열로만 답하세요. "
# 위험도 세 단어와 요약 길이를 정한다. 뒤에 경보 목록이 붙는다
instruction = instruction + "risk_level 은 high, medium, low 중 하나입니다. summary 는 한국어 한 문장으로 짧게 씁니다. 경보 목록: "


def summarize_batch(batch):                                   # 묶음 하나를 LLM 으로 요약하는 함수
    # 묶음을 JSON 문자열로 바꿔 지시와 함께 보낸다
    text = call_llm(instruction + json.dumps(batch, ensure_ascii=False))
    result = parse_llm_json(text)                             # 답을 딕셔너리로 바꾼다. 못 읽으면 None
    if result is None:                                        # JSON 으로 읽지 못했으면
        return []                                             # 실패하면 빈 리스트 — 다음 묶음은 계속한다
    return result                                             # 요약 리스트를 돌려준다


def summarize_events(events):                                 # 경보 전체를 묶음으로 나눠 요약하는 함수
    summaries = []                                            # 요약을 모을 빈 리스트
    for start in range(0, len(events), SIZE):                 # 0, 6, 12 … 묶음의 시작 번호
        # 묶음 하나를 요약해 한 건씩 꺼낸다
        for item in summarize_batch(events[start:start + SIZE]):
            summaries.append(item)                            # 요약 하나를 모은다
    return summaries                                          # 모은 요약을 돌려준다


def sort_by_risk(summaries):                                  # 요약을 high → medium → low 순서로 정렬하는 함수
    sorted_summaries = []                                     # 위험도순으로 모을 빈 리스트
    for level in ["high", "medium", "low"]:                   # 이 순서대로 모은다
        for s in summaries:                                   # 요약을 하나씩 꺼낸다
            if s["risk_level"].lower() == level:              # 소문자로 맞춰 지금 위험도와 같으면
                sorted_summaries.append(s)                    # 정렬 결과에 넣는다
    return sorted_summaries                                   # 정렬한 요약을 돌려준다

Overwriting event_summarizer.py


In [5]:
# 준비 셀 — 그대로 실행합니다. 오전의 요약 결과(정렬한 18건)와 내일 쓸 경보 6건을 파일로 저장합니다
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

sorted_summaries = [                                          # 오전에 위험도순으로 정렬한 결과라고 가정합니다
    {"id": "E01", "risk_level": "high", "summary": "admin 계정에 로그인 실패 4회"},
    {"id": "E02", "risk_level": "High", "summary": "실패 직후 admin 로그인 성공 — 탈취 의심"},
    {"id": "E03", "risk_level": "high", "summary": "한 IP 의 비밀번호 스프레잉"},
    {"id": "E04", "risk_level": "high", "summary": "root 계정에 로그인 실패 6회"},
    {"id": "E12", "risk_level": "high", "summary": "guest01 의 shadow 접근 거부"},
    {"id": "E16", "risk_level": "HIGH", "summary": "관리자 권한 계정 무단 생성 의심"},
    {"id": "E05", "risk_level": "medium", "summary": "svc_backup 의 sudoers 접근 거부"},
    {"id": "E08", "risk_level": "medium", "summary": "deploy 토큰 만료"},
    {"id": "E09", "risk_level": "medium", "summary": "guest 계정에 로그인 실패 5회"},
    {"id": "E10", "risk_level": "medium", "summary": "lee.yh 새 IP 로그인"},
    {"id": "E13", "risk_level": "Medium", "summary": "외부 IP 의 포트 스캔"},
    {"id": "E15", "risk_level": "medium", "summary": "svc_batch 새벽 로그인 실패"},
    {"id": "E06", "risk_level": "low", "summary": "kim01 로그인 실패 1회"},
    {"id": "E07", "risk_level": "low", "summary": "park.js 실패 뒤 정상 로그인"},
    {"id": "E11", "risk_level": "low", "summary": "choi.mk 로그인 실패 2회"},
    {"id": "E14", "risk_level": "low", "summary": "jung.hw 로그인 실패 1회"},
    {"id": "E17", "risk_level": "low", "summary": "kim.cs 로그인 실패 1회"},
    {"id": "E18", "risk_level": "low", "summary": "점검 중 방화벽 규칙 수정"},
]

events_1008 = [                                               # 7교시 「다른 날」로 쓸 경보 6건
    {"id": "F01", "rule": "brute_force", "target": "admin", "time": "02:10", "detail": "203.0.113.9 에서 admin 로그인 실패 7회 뒤 성공"},
    {"id": "F02", "rule": "failed_login", "target": "kim01", "time": "09:05", "detail": "kim01 로그인 실패 1회"},
    {"id": "F03", "rule": "failed_login", "target": "lee.yh", "time": "09:40", "detail": "lee.yh 로그인 실패 1회 뒤 바로 성공"},
    {"id": "F04", "rule": "config_changed", "target": "firewall", "time": "18:00", "detail": "점검 시간에 ops.kim 이 방화벽 규칙 1개 수정"},
    {"id": "F05", "rule": "failed_login", "target": "park.js", "time": "10:12", "detail": "park.js 로그인 실패 1회"},
    {"id": "F06", "rule": "new_ip_login", "target": "jung.hw", "time": "08:55", "detail": "jung.hw 가 사내 다른 층 IP 에서 로그인"},
]

with open("sorted_summaries.json", "w", encoding="utf-8") as f:   # 정렬한 요약 — 오후 내내 쓴다
    # 정렬한 요약을 파일에 저장한다
    json.dump(sorted_summaries, f, ensure_ascii=False, indent=2)
with open("events_1008.json", "w", encoding="utf-8") as f:        # 7교시에 「다른 날」로 쓴다
    json.dump(events_1008, f, ensure_ascii=False, indent=2)   # 내일 경보를 파일에 저장한다

print(f"정렬한 요약 {len(sorted_summaries)}건 · 내일 경보 {len(events_1008)}건을 저장했습니다")

정렬한 요약 18건 · 내일 경보 6건을 저장했습니다


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">5교시 (14:00–14:50) · 요약을 다시 요약하기 — 총평</mark>

## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **프롬프트 체이닝 (prompt chaining)** | 한 번에 다 묻지 않고 단계를 나눠 묻는 이유 |
| **경영진 요약 (executive summary)** | 보고서 맨 앞의 요약에는 무엇을 쓰나 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다.

- 프롬프트 체이닝 →
- 경영진 요약 →

### 왜 필요한가

1. 건별 요약 18줄을 팀장이 다 읽기는 어렵습니다. **두세 문장짜리 총평**이 맨 앞에 있어야 합니다.
2. 총평은 원래 경보가 아니라 **오전에 만든 요약**을 모아 다시 LLM 에 보냅니다. 이미 짧고 정리돼 있어서 LLM 이 핵심을 잘 잡습니다.
3. 이렇게 앞 단계의 답을 다음 단계의 질문에 넣는 것을 **프롬프트 체이닝**이라고 합니다. 오전의 묶음 요약이 1단계, 총평이 2단계입니다.

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.1 요약 18줄을 글 하나로 모아 총평을 받습니다</mark>

```python
summary_text = ""
for s in sorted_summaries:
    summary_text = summary_text + f"- [{s['risk_level'].upper()}] {s['summary']}\n"

overview = call_llm(overview_instruction + summary_text)
```

| 단계 | 넣는 것 | 받는 것 |
|---|---|---|
| 1단계 (오전) | 원래 경보 6건씩 | 건별 요약 — `id` · `risk_level` · `summary` |
| 2단계 (지금) | 건별 요약 18줄 | 총평 두세 문장 |

지시에 **「숫자는 쓰지 마세요」** 를 넣습니다. 건수는 6교시에 코드가 셉니다.

#### 🐍 문법 상자 · 리스트를 문자열 하나로 모으기

```python
summaries = [{"id": "E01", "summary": "로그인 실패"}, {"id": "E02", "summary": "로그인 성공"}]
text = ""
for s in summaries:
    text = text + s["id"] + " " + s["summary"] + "\n"
print(len(text.split("\n")))
# 3
```

| 쓰는 것 | 하는 일 |
|---|---|
| `text = ""` | 빈 문자열에서 시작 |
| `text = text + … + "\n"` | 한 줄씩 뒤에 붙인다 |
| `text.split("\n")` | 줄바꿈 자리에서 다시 나눠 리스트로 |

⚠ 마지막 줄 뒤에도 `\n` 이 있어서 나누면 빈 칸 하나가 더 생깁니다. 두 줄인데 3이 나오는 이유입니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
sorted_summaries = [{"risk_level": "high", "summary": "로그인 실패 4회"},
                    {"risk_level": "low", "summary": "점검 중 설정 변경"}]

summary_text = ""
for s in sorted_summaries:
    summary_text = summary_text + "- [" + s["risk_level"].upper() + "] " + s["summary"] + "\n"

print(summary_text)
```

막히면 바로 위 `5.1 요약 18줄을 글 하나로 모아 총평을 받습니다` 설명을 다시 봅니다.

In [6]:
sorted_summaries = [{"risk_level": "high", "summary": "로그인 실패 4회"},
                    {"risk_level": "low", "summary": "점검 중 설정 변경"}]

summary_text = ""
for s in sorted_summaries:
    summary_text = summary_text + "- [" + s["risk_level"].upper() + "] " + s["summary"] + "\n"

print(summary_text)

- [HIGH] 로그인 실패 4회
- [LOW] 점검 중 설정 변경



✅ `- [HIGH] 로그인 실패 4회` · `- [LOW] 점검 중 설정 변경` 두 줄

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-2 · 총평에 넣을 요약 글 만들기</font></h3></td></tr></table>

정렬한 요약 18건을 `- [HIGH] 요약` 꼴의 줄로 이어 붙여 `summary_text` 를 만드시오. LLM 은 아직 부르지 않습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `- [HIGH] admin 계정에 로그인 실패 4회` 로 시작하는 앞부분과 전체 글자 수 |

**💡 힌트**

1. 오전 문제 4-4 와 같은 모양입니다. 이번에는 id 를 넣지 않습니다.
2. 위험도는 `.upper()` 로 대문자로 맞춥니다.
3. 줄 끝의 `\n` 을 빠뜨리지 않습니다.

In [7]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("sorted_summaries.json", encoding="utf-8") as f:   # 준비 셀이 저장한 정렬된 요약
    sorted_summaries = json.load(f)                           # 정렬된 요약 18건을 리스트로 읽는다
summary_text = ""                                             # LLM 에 보낼 요약 글
# ── 여기까지 ──

# 1. for 로 sorted_summaries 에서 하나씩 꺼내 s 에 담으세요
for s in sorted_summaries:
    # 2. summary_text 뒤에 f"- [{s['risk_level'].upper()}] {s['summary']}\n" 을 이어 붙이세요
    summary_text += f"- [{s['risk_level'].upper()}] {s['summary']}\n"

print(summary_text[0:120])                                    # 앞부분만 출력한다
print("…", len(summary_text), "글자")

- [HIGH] admin 계정에 로그인 실패 4회
- [HIGH] 실패 직후 admin 로그인 성공 — 탈취 의심
- [HIGH] 한 IP 의 비밀번호 스프레잉
- [HIGH] root 계정에 로그인 실패 6회
-
… 507 글자


#### 🐍 문법 상자 · `.strip()` — LLM 답의 앞뒤 공백 지우기

```python
answer = "\n  관리자 계정 탈취가 의심됩니다.  \n"
print(len(answer), len(answer.strip()))
# 23 17
```

| 쓰는 것 | 결과 |
|---|---|
| `answer` | 앞뒤에 빈칸 · 줄바꿈이 붙어 있을 수 있다 |
| `answer.strip()` | 앞뒤만 지운다. 문장 가운데 빈칸은 그대로 |

⚠ LLM 의 답은 끝에 줄바꿈이 붙어 오는 일이 많습니다. 보고서에 넣기 전에 `.strip()` 합니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-3 · LLM 에게 총평 받기</font></h3></td></tr></table>

`summary_text` 를 LLM 에 보내 총평을 받으시오. **LLM 을 한 번 부릅니다.**

| | |
|---|---|
| 🎯 나와야 하는 결과 | 두세 문장의 한국어 총평. 숫자가 없어야 합니다. 문장은 실행할 때마다 조금 다릅니다 |

**💡 힌트**

1. 질문은 `overview_instruction + summary_text` 입니다.
2. 받은 답은 `.strip()` 으로 앞뒤를 다듬습니다.
3. 빈 줄만 나오면 한도에 걸린 것입니다. 1분 뒤 다시 실행합니다.

In [8]:
import json                                                # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import importlib                                           # 파일을 다시 읽는 도구
import llm_client                                          # 오전에 만든 llm_client.py 를 가져옴

importlib.reload(llm_client)                              # llm_client.py 를 고쳤으면 다시 읽음
from llm_client import call_llm                            # 10/6 에 만든 함수를 꺼내 쓴다

# — 미리 채워 둔 줄입니다. 고치지 않습니다 —
with open("sorted_summaries.json", encoding="utf-8") as f: # 준비 셀이 저장한 정렬된 요약
    sorted_summaries = json.load(f)                        # 정렬된 요약 18건을 리스트로 읽는다
summary_text = ""                                          # LLM 에 보낼 요약 글
for s in sorted_summaries:                                 # 정렬된 요약을 하나씩 꺼낸다
    # 요약 한 줄을 붙인다
    summary_text = summary_text + f"- [{s['risk_level'].upper()}] {s['summary']}\n"
# 총평 지시 — 숫자는 코드가 세므로 쓰지 말라고 한다
overview_instruction = "다음은 밤사이 보안 경보 요약입니다. 팀장에게 보고할 총평을 한국어 두세 문장으로 작성하세요. 숫자는 포함하지 마세요.\n\n"
# — 여기까지 —

# 1. overview 라는 변수에 call_llm(overview_instruction + summary_text) 의 답을 담으세요
overview = call_llm(overview_instruction + summary_text)
# 2. overview 의 앞뒤 공백을 지운 값을 다시 overview 에 담으세요 (.strip())
overview = overview.strip()

print(overview)                                            # 총평을 본다

밤사이 관리자 계정을 겨냥한 무차별 대입 공격과 무단 계정 생성 정황이 포착되는 등 심각한 수준의 침해 시도가 발생했습니다. 외부의 포트 스캔과 함께 일부 권한 상승 및 접근 거부 시도도 확인되어 긴급한 보안 조치와 로그 분석이 필요합니다.


#### 🐍 문법 상자 · 빈 문자열은 「답이 없다」는 뜻

```python
overview = ""
print(overview == "", len(overview))
# True 0
```

| 값 | `== ""` | 뜻 |
|---|---|---|
| `""` | `True` | 아무 글자도 없다 — `call_llm` 이 실패하면 이 값을 돌려준다 |
| `" "` | `False` | 빈칸 한 글자가 있다 |

⚠ `call_llm` 은 한도 초과(429) 같은 실패 때 `""` 를 돌려줍니다. 그대로 보고서에 넣으면 총평 칸이 비어 버립니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-4 · 총평을 못 받았을 때 대신 쓸 문장</font></h3></td></tr></table>

`call_llm` 이 실패하면 빈 문자열 `""` 을 돌려줍니다. 그때 보고서의 총평 칸이 비지 않도록 대신 쓸 문장을 넣으시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `총평을 만들지 못했습니다. 건별 내역을 먼저 확인하세요.` |

**💡 힌트**

1. 빈 문자열인지는 `== ""` 로 확인합니다.
2. 실패를 숨기지 않고 「못 만들었다」고 적습니다. 보고서를 읽는 사람이 알아야 합니다.
3. LLM 을 부르지 않습니다. 실패한 경우를 가정한 문제입니다.

In [9]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
overview = ""                                                 # LLM 이 답을 못 준 경우라고 가정합니다
# ── 여기까지 ──

# 1. overview 가 빈 문자열이면 (if 문)
overview = overview.strip()
if not overview:
    # 2. overview 에 "총평을 만들지 못했습니다. 건별 내역을 먼저 확인하세요." 를 담으세요
    overview = "총평을 만들지 못했습니다. 건별 내역을 먼저 확인하세요."

print(overview)                                               # 총평을 본다

총평을 만들지 못했습니다. 건별 내역을 먼저 확인하세요.


#### 🐍 문법 상자 · 함수 안의 `return` 은 거기서 끝낸다

```python
def check(text):
    if text == "":
        return "대신 쓸 문장"
    return text.strip()
print(check(""))
print(check(" 총평 "))
# 대신 쓸 문장
# 총평
```

| 들어온 값 | 실행되는 줄 | 돌려주는 값 |
|---|---|---|
| `""` | 첫 `return` 에서 끝난다 | `"대신 쓸 문장"` |
| `" 총평 "` | `if` 를 건너뛰고 두 번째 `return` | `"총평"` |

⚠ 첫 `return` 을 만나면 아래 줄은 실행되지 않습니다. 그래서 `else` 없이도 두 갈래가 됩니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-5 · make_overview 함수로 묶기</font></h3></td></tr></table>

요약 글 만들기 · LLM 호출 · 실패 대비를 함수 `make_overview` 하나로 묶으시오. **LLM 을 한 번 부릅니다.**

| | |
|---|---|
| 🎯 나와야 하는 결과 | 두세 문장의 한국어 총평 |

**💡 힌트**

1. 요약 글을 만드는 반복은 미리 채워져 있습니다.
2. 실패하면 그 자리에서 `return` 합니다. 문제 5-4 와 같은 문장입니다.
3. 마지막 `return` 에서 `.strip()` 합니다.

In [10]:
import json                                                # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import importlib                                           # 파일을 다시 읽는 도구
import llm_client                                          # 오전에 만든 llm_client.py 를 불러온다

importlib.reload(llm_client)                              # llm_client.py 를 고쳤으면 다시 읽습니다
from llm_client import call_llm                            # 10/6 에 만든 함수를 꺼내 쓴다

# — 미리 채워 둔 줄입니다. 고치지 않습니다 —
with open("sorted_summaries.json", encoding="utf-8") as f: # 준비 셀이 저장한 정렬된 요약
    sorted_summaries = json.load(f)                        # 정렬된 요약 18건을 리스트로 읽는다
# — 여기까지 —

# 1. risk_level 이 high 인 요약만 모아 high_text 라는 문자열로 만드세요 (한 줄에 하나, .lower())
high_text = "\n".join([f"- [{s['risk_level'].upper()}] {s['summary']}" for s in sorted_summaries if s['risk_level'].lower() == 'high'])

# 2. "다음 high 경보를 보고, 보안 담당자가 가장 먼저 할 조치 한 가지를 한국어 한 문장으로 쓰세요: " 뒤에 high_text 를 붙여 LLM 에 보내세요
guidance = "다음 high 경보를 보고, 보안 담당자가 가장 먼저 할 조치 한 가지를 한국어 한 문장으로 쓰세요: " + high_text

# 3. 받은 답을 "먼저 할 일:" 과 함께 출력하세요
high_guidance = call_llm(guidance)
print("먼저 할 일:", high_guidance.strip())

먼저 할 일: 보안 담당자는 추가 피해를 막기 위해 탈취가 의심되는 **admin 계정을 즉시 차단(잠금)하고 세션을 강제 종료**해야 합니다.


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>

<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 5-6 · high 만 골라 「먼저 할 일」 받기</font></h3></td></tr></table>

high 요약만 모아 LLM 에 보내고, 보안 담당자가 **가장 먼저 할 조치 한 가지**를 받으시오. **LLM 을 한 번 부릅니다.**

| | |
|---|---|
| 🎯 나와야 하는 결과 | `먼저 할 일: admin 계정을 잠그고 …` 꼴의 한 문장 |

**💡 힌트**

1. high 만 고르는 것은 오전 문제 4-2 와 같습니다.
2. 체이닝을 한 단계 더 한 것입니다 — 요약 → high 만 → 조치.
3. LLM 의 조치 제안은 **참고용**입니다. 실제 조치는 10/6 의 승인 게이트처럼 사람이 정합니다.

In [11]:
# — 미리 채워 둔 줄입니다. 고치지 않습니다 —
overview = ""                                            # LLM 이 답을 못 준 경우라고 가정합니다
# — 여기까지 —

# 1. overview 가 빈 문자열이면 (if 문)
if overview == "":

    # 2. overview 에 "총평을 만들지 못했습니다. 건별 내역을 먼저 확인하세요." 를 담으세요
    overview = "총평을 만들지 못했습니다. 건별 내역을 먼저 확인하세요."

print(overview)                                          # 총평을 본다

총평을 만들지 못했습니다. 건별 내역을 먼저 확인하세요.


### 📋 5교시 한눈에

| 하려는 일 | 쓰는 코드 |
|---|---|
| 요약을 글 하나로 모은다 | `summary_text = summary_text + f"- [{…upper()}] {…}\n"` |
| 총평을 받는다 | `call_llm(overview_instruction + summary_text).strip()` |
| 실패를 대비한다 | `if text == "": return "총평을 만들지 못했습니다. …"` |
| 묶는다 | `make_overview(sorted_summaries)` |

앞 단계의 답을 다음 질문에 넣는 것이 프롬프트 체이닝입니다. 숫자는 맡기지 않습니다.

---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">6교시 (15:00–15:50) · 보고서 틀 — 숫자는 코드가, 문장은 LLM 이</mark>

## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **템플릿 (template)** | 매번 같은 모양의 문서를 만들 때 무엇을 고정하나 |
| **조건부 서식** | 값에 따라 문구나 색을 바꾸는 것은 언제 쓰나 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다.

- 템플릿 →
- 조건부 서식 →

### 왜 필요한가

1. 보고서 전체를 LLM 에게 쓰게 하면 날마다 소제목 · 순서 · 모양이 달라집니다. 어제와 오늘을 비교할 수 없습니다.
2. LLM 은 **건수를 틀리게 셀 수 있습니다.** 「18건」이 「17건」이 되면 보고서를 믿을 수 없습니다.
3. 그래서 **틀(제목 · 소제목 · 순서)과 숫자는 코드가**, **총평 문장만 LLM 이** 씁니다. high 가 3건 이상이면 맨 위에 경고 줄을 넣는 것도 코드가 정합니다.

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.1 보고서는 네 덩어리를 순서대로 이어 붙입니다</mark>

```python
report = f"# 야간 보안 관제 보고 ({today})\n\n"
report = report + "## 한눈에 보기\n" + summary_line + warning
report = report + "\n## 총평\n" + overview + "\n"
report = report + "\n## 건별 내역 (위험한 것부터)\n" + lines
```

| 덩어리 | 만드는 곳 | 문제 |
|---|---|---|
| `summary_line` — 건수 | 코드 | 6-2 |
| `warning` — 조건부 경고 | 코드 | 6-3 |
| `overview` — 총평 | LLM (5교시) | — |
| 넷을 잇기 · 저장 | 코드 | 6-4 · 6-5 |

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
high_count = 4
warning = ""
if high_count >= 3:
    warning = "- **주의**\n"

print("## 한눈에 보기\n" + warning + "- 끝")
```

막히면 바로 위 `6.1 보고서는 네 덩어리를 순서대로 이어 붙입니다` 설명을 다시 봅니다.

In [12]:
high_count = 4
warning = ""
if high_count >= 3:
    warning = "- **주의**\n"

print("## 한눈에 보기\n" + warning + "- 끝")

## 한눈에 보기
- **주의**
- 끝


✅ `## 한눈에 보기` · `- **주의**` · `- 끝` 세 줄 — high 가 3 이상이라 경고 줄이 들어갑니다

#### 🐍 문법 상자 · 숫자는 코드가, 문장은 LLM 이

```python
summaries = [{"risk_level": "high"}, {"risk_level": "High"}, {"risk_level": "low"}]
high = 0
for s in summaries:
    if s["risk_level"].lower() == "high":
        high = high + 1
print(len(summaries), high)
# 3 2
```

| 보고서의 칸 | 누가 쓰나 | 이유 |
|---|---|---|
| 건수 · high 개수 | **코드** | 세는 것은 틀리면 안 된다. LLM 은 숫자를 틀리게 셀 수 있다 |
| 총평 문장 | **LLM** | 사람이 읽기 좋은 문장은 LLM 이 잘 쓴다 |
| 제목 · 소제목 순서 | **코드** | 매일 같은 모양이어야 비교할 수 있다 |

⚠ 총평을 부탁할 때 「숫자는 쓰지 마세요」라고 적은 이유입니다. 숫자는 코드가 센 값만 보고서에 들어갑니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-2 · 건수 줄 만들기 — 숫자는 코드가</font></h3></td></tr></table>

전체 건수와 high 건수를 코드로 세어 `- 처리한 경보: 18건 (high 6건)` 줄을 만드시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `- 처리한 경보: 18건 (high 6건)` |

**💡 힌트**

1. 전체 건수는 `len()` 입니다.
2. high 세기는 오전 문제 4-2 와 같습니다. `.lower()` 를 잊지 않습니다.
3. 줄 끝에 `\n` 을 붙입니다.

In [13]:
import json                                                # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# — 미리 채워 둔 줄입니다. 고치지 않습니다 —
with open("sorted_summaries.json", encoding="utf-8") as f: # 준비 셀이 저장한 정렬된 요약
    sorted_summaries = json.load(f)                        # 정렬된 요약 18건을 리스트로 읽는다
# — 여기까지 —

high_count = 0                                             # high 건수. 0 에서 시작

# 1. total 이라는 변수에 sorted_summaries 의 길이를 담으세요
total = len(sorted_summaries)

# 2. for 로 sorted_summaries 를 돌면서 risk_level 을 소문자로 바꾼 값이 "high" 이면 high_count 에 1 을 더하세요
for s in sorted_summaries:
    if s["risk_level"].lower() == "high":
        high_count += 1

# 3. summary_line 에 f"- 처리한 경보: {total}건 (high {high_count}건)\n" 을 담으세요
summary_line = f"- 처리한 경보: {total}건 (high {high_count}건)\n"

print(summary_line)

- 처리한 경보: 18건 (high 6건)



#### 🐍 문법 상자 · 조건에 따라 문구를 넣거나 빼기

```python
high_count = 2
warning = ""
if high_count >= 3:
    warning = "주의\n"
print("[" + warning + "]")
# []
```

| `high_count` | `warning` | 보고서에서 |
|---|---|---|
| 3 이상 | `"- **주의: …**\n"` | 「한눈에 보기」 아래 경고 줄이 생긴다 |
| 2 이하 | `""` (빈 문자열) | 이어 붙여도 아무것도 생기지 않는다 |

⚠ `warning` 을 `""` 로 먼저 만들어 두어야 조건이 거짓일 때도 이어 붙일 수 있습니다. 없으면 `NameError` 가 납니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-3 · high 가 3건 이상이면 경고 줄 넣기</font></h3></td></tr></table>

high 건수가 3 이상이면 경고 줄을, 아니면 빈 문자열을 `warning` 에 담으시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `- **주의: high 경보 6건 — 건별 내역을 먼저 확인할 것**` 과 `경고 줄 글자 수: 39` |

**💡 힌트**

1. `warning = ""` 는 미리 채워져 있습니다. 조건이 맞을 때만 바꿉니다.
2. 비교는 `>=` 입니다. 3건이면 경고가 들어갑니다.
3. `high_count` 를 2 로 바꿔 실행하면 글자 수가 0 이 되는지 봅니다.

In [14]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
high_count = 6                                                # 문제 6-2 에서 센 값
# ── 여기까지 ──

warning = ""                                                  # 기본은 경고 없음

# 1. high_count 가 3 이상이면 (if 문)
if high_count >= 3:
    # 2. warning 에 f"- **주의: high 경보 {high_count}건 — 건별 내역을 먼저 확인할 것**\n" 을 담으세요
    warning = f"- **주의: high 경보 {high_count}건 — 건별 내역을 먼저 확인할 것**\n"

print(warning)
print("경고 줄 글자 수:", len(warning))

- **주의: high 경보 6건 — 건별 내역을 먼저 확인할 것**

경고 줄 글자 수: 39


#### 🐍 문법 상자 · 보고서의 틀은 코드에 고정합니다

```python
title = "# 보고\n\n"
part1 = "## 한눈에 보기\n- 18건\n"
part2 = "\n## 총평\n문장\n"
report = title + part1 + part2
print(report.count("## "))
# 2
```

| 순서 | 소제목 | 채우는 것 |
|---|---|---|
| 1 | `# 야간 보안 관제 보고 (날짜)` | 날짜 — 코드 |
| 2 | `## 한눈에 보기` | 건수 · 경고 — 코드 |
| 3 | `## 총평` | 문장 — LLM |
| 4 | `## 건별 내역 (위험한 것부터)` | 정렬한 요약 — 코드 |

⚠ 보고서 전체를 LLM 에게 쓰게 하면 날마다 소제목과 순서가 바뀝니다. 틀을 코드에 고정하면 어제와 오늘 보고서를 나란히 비교할 수 있습니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-4 · 네 덩어리를 이어 보고서 만들기</font></h3></td></tr></table>

제목 · 한눈에 보기 · 총평 · 건별 내역을 순서대로 이어 붙여 `report` 를 만드시오. 총평은 5교시에 받았다고 가정한 문장이 미리 들어 있습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `# 야간 보안 관제 보고 (2026-10-07)` 로 시작해 `## 한눈에 보기` · `## 총평` · `## 건별 내역` 이 차례로 보이는 앞부분 |

**💡 힌트**

1. 한 번에 다 쓰지 않고 네 줄로 나눠 이어 붙입니다.
2. 소제목 앞의 `\n` 은 위 덩어리와 빈 줄을 두기 위한 것입니다.
3. 순서를 바꾸지 않습니다. 매일 같은 순서여야 비교할 수 있습니다.

In [15]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("sorted_summaries.json", encoding="utf-8") as f:   # 준비 셀이 저장한 정렬된 요약
    sorted_summaries = json.load(f)                           # 정렬된 요약 18건을 리스트로 읽는다
today = "2026-10-07"                                          # 보고서 날짜
summary_line = "- 처리한 경보: 18건 (high 6건)\n"                   # 문제 6-2 의 결과
warning = "- **주의: high 경보 6건 — 건별 내역을 먼저 확인할 것**\n"  # 문제 6-3 의 결과
overview = "밤사이 관리자 계정을 노린 무차별 대입과 로그인 성공이 이어져 계정 탈취가 의심됩니다. 관리자 권한 계정이 새로 만들어진 정황도 있어 우선 확인이 필요합니다."      # 5교시에 LLM 이 쓴 총평이라고 가정합니다
lines = ""                                                    # 건별 내역 줄
for s in sorted_summaries:                                    # 정렬된 요약을 하나씩 꺼낸다
    # 「- [HIGH] E01 요약」 한 줄을 붙인다
    lines = lines + f"- [{s['risk_level'].upper()}] {s['id']} {s['summary']}\n"
# ── 여기까지 ──

# 1. report 에 f"# 야간 보안 관제 보고 ({today})\n\n" 을 담으세요
report = f"# 야간 보안 관제 보고 ({today})\n\n"

# 2. report 뒤에 "## 한눈에 보기\n" + summary_line + warning 을 이어 붙이세요
report += "## 한눈에 보기\n" + summary_line + warning

# 3. report 뒤에 "\n## 총평\n" + overview + "\n" 을 이어 붙이세요
report += "\n## 총평\n" + overview + "\n"

# 4. report 뒤에 "\n## 건별 내역 (위험한 것부터)\n" + lines 를 이어 붙이세요
report += "\n## 건별 내역 (위험한 것부터)\n" + lines

print(report[0:300])

# 야간 보안 관제 보고 (2026-10-07)

## 한눈에 보기
- 처리한 경보: 18건 (high 6건)
- **주의: high 경보 6건 — 건별 내역을 먼저 확인할 것**

## 총평
밤사이 관리자 계정을 노린 무차별 대입과 로그인 성공이 이어져 계정 탈취가 의심됩니다. 관리자 권한 계정이 새로 만들어진 정황도 있어 우선 확인이 필요합니다.

## 건별 내역 (위험한 것부터)
- [HIGH] E01 admin 계정에 로그인 실패 4회
- [HIGH] E02 실패 직후 admin 로그인 성공 — 탈취 의심
- [HIGH] E


#### 🐍 문법 상자 · `.startswith()` 와 `.split("\n")`

```python
text = "# 제목\n- 내용\n## 소제목"
for line in text.split("\n"):
    if line.startswith("#"):
        print(line)
# # 제목
# ## 소제목
```

| 쓰는 것 | 결과 |
|---|---|
| `text.split("\n")` | 줄마다 나눈 리스트 |
| `line.startswith("#")` | 그 줄이 `#` 으로 시작하면 `True` |
| `f.read()` | 파일 전체를 문자열 하나로 읽는다 |

⚠ `"## 소제목".startswith("#")` 도 `True` 입니다. 소제목만 고르려면 `"## "` 로 비교합니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-5 · 보고서를 저장하고 제목 줄만 확인하기</font></h3></td></tr></table>

보고서를 날짜가 든 파일 이름으로 저장하고, 다시 읽어 `#` 으로 시작하는 줄만 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `# 야간 보안 관제 보고 (2026-10-07)` · `## 한눈에 보기` · `## 총평` · `## 건별 내역 (위험한 것부터)` 네 줄 |

**💡 힌트**

1. 파일 이름은 `daily_report_20261007.md` 가 됩니다.
2. 쓰기는 `"w"`, 읽기는 모드를 적지 않습니다.
3. 왼쪽 목록에서 파일을 열고 `Ctrl+Shift+V` 로 미리 보기를 해 봅니다.

In [16]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
today = "2026-10-07"                                          # 보고서 날짜
report = f"# 야간 보안 관제 보고 ({today})\n\n## 한눈에 보기\n- 처리한 경보: 18건 (high 6건)\n"      # 문제 6-4 의 결과를 줄였습니다
report = report + "- **주의: high 경보 6건 — 건별 내역을 먼저 확인할 것**\n\n## 총평\n총평 문장\n\n## 건별 내역 (위험한 것부터)\n- [HIGH] E01 admin 계정에 로그인 실패 4회\n"
# ── 여기까지 ──

# 1. filename 에 f"daily_report_{today.replace('-', '')}.md" 를 담으세요
filename = f"daily_report_{today.replace('-', '')}.md"

# 2. filename 을 쓰기로 열고(encoding="utf-8") report 를 저장하세요 (with 문, f.write)

with open(filename, "w", encoding="utf-8") as f:
    f.write(report)

# 3. filename 을 읽기로 열고 내용을 text 에 담으세요 (f.read())
with open(filename,  "r", encoding="utf-8") as f:
    text = f.read()

for line in text.split("\n"):                                 # 한 줄씩 나눠
    if line.startswith("#"):                                   # # 으로 시작하는 줄(제목)만
        print(line)

# 야간 보안 관제 보고 (2026-10-07)
## 한눈에 보기
## 총평
## 건별 내역 (위험한 것부터)


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>

<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 6-6 · build_report 함수로 묶고 다른 데이터로 돌리기</font></h3></td></tr></table>

6-2 ~ 6-4 의 줄을 함수 `build_report` 하나로 묶고, high 가 2건뿐인 데이터로 실행해 **경고 줄이 빠지는지** 확인하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `- 처리한 경보: 4건 (high 2건)` 아래에 `**주의` 줄이 없는 보고서 |

**💡 힌트**

1. 함수의 입력은 정렬한 요약 · 총평 · 날짜 셋입니다.
2. 함수 안에서 `total` · `high_count` · `warning` · `lines` 를 차례로 만듭니다.
3. 이 함수가 7교시 `report_generator.py` 의 중심이 됩니다.

In [17]:
# — 미리 채워 둔 줄입니다. 고치지 않습니다 —
small = [                                                  # high 가 2건뿐인 작은 데이터
    {"id": "E01", "risk_level": "high", "summary": "admin 계정에 로그인 실패 4회"},
    {"id": "E02", "risk_level": "High", "summary": "실패 직후 admin 로그인 성공 – 탈취 의심"},
    {"id": "E05", "risk_level": "medium", "summary": "svc_backup 의 sudoers 접근 거부"},
    {"id": "E06", "risk_level": "low", "summary": "kim01 로그인 실패 1회"},
]
# — 여기까지 —

# 1. build_report(sorted_summaries, overview, today) 함수를 만드세요. 문제 6-2 ~ 6-4 의 줄을 함수 안에 모아 report 를 돌려줍니다
def build_report(sorted_summaries, overview, today):
    # 건수 세기 (문제 6-2)
    total = len(sorted_summaries)
    high_count = 0
    for s in sorted_summaries:
        if s["risk_level"].lower() == "high":
            high_count += 1
    summary_line = f"- 처리한 경보: {total}건 (high {high_count}건)\n"
    
    # 경고 줄 생성 (문제 6-3)
    warning = ""
    if high_count >= 3:
        warning = f"- **주의: high 경보 {high_count}건 – 건별 내역을 먼저 확인할 것**\n"
        
    # 건별 내역 생성 (문제 6-4)
    lines = ""
    for s in sorted_summaries:
        lines += f"- [{s['risk_level'].upper()}] {s['id']} {s['summary']}\n"
        
    # 전체 보고서 조립 (문제 6-4)
    report = f"# 야간 보안 관제 보고 ({today})\n\n"
    report += "## 한눈에 보기\n" + summary_line + warning
    report += "\n## 총평\n" + overview + "\n"
    report += "\n## 건별 내역 (위험한 것부터)\n" + lines
    
    return report

# 2. small(high 2건)로 보고서를 만들어 출력하세요. overview 는 "총평 문장", today 는 "2026-10-08" 입니다
result_report = build_report(small, "총평 문장", "2026-10-08")
print(result_report)

# 야간 보안 관제 보고 (2026-10-08)

## 한눈에 보기
- 처리한 경보: 4건 (high 2건)

## 총평
총평 문장

## 건별 내역 (위험한 것부터)
- [HIGH] E01 admin 계정에 로그인 실패 4회
- [HIGH] E02 실패 직후 admin 로그인 성공 – 탈취 의심
- [MEDIUM] E05 svc_backup 의 sudoers 접근 거부
- [LOW] E06 kim01 로그인 실패 1회



### 📋 6교시 한눈에

| 보고서의 칸 | 누가 | 쓰는 코드 |
|---|---|---|
| 제목 · 소제목 순서 | 코드 | `report = report + "## …\n" + …` |
| 건수 | 코드 | `len(sorted_summaries)` · `high_count` |
| 경고 줄 | 코드 | `if high_count >= 3: warning = …` |
| 총평 | LLM | `make_overview(…)` |
| 저장 | 코드 | `daily_report_{날짜}.md` |

---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">7교시 (16:00–16:50) · report_generator.py — 처음부터 끝까지 잇기</mark>

## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **파이프라인 (pipeline)** | 데이터를 여러 단계로 차례로 처리하는 구조 |
| **재현성** | 같은 코드를 다른 날 데이터로 돌려도 같은 모양이 나와야 하는 이유 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다.

- 파이프라인 →
- 재현성 →

### 왜 필요한가

1. 지금까지 만든 것을 **파일 두 개**로 모읍니다. 요약 · 정렬은 `event_summarizer.py`(오전), 총평 · 보고서는 `report_generator.py`(지금).
2. 파일로 묶어 두면 **내일 경보로 그대로** 돌릴 수 있습니다. 7교시에 실제로 「다른 날」 경보 6건으로 다시 돌려 봅니다.
3. 학원 평가 기준이 여기서 다 확인됩니다 — 묶음 처리와 정렬 · 보고서 형식이 매번 같음 · high 3건 이상이면 경고.

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">7.1 함수 넷을 차례로 부르면 보고서가 나옵니다</mark>

```python
summaries = event_summarizer.summarize_events(events)        # 1. 묶음 요약 (LLM)
sorted_summaries = event_summarizer.sort_by_risk(summaries)  # 2. 위험도순
overview = report_generator.make_overview(sorted_summaries)  # 3. 총평 (LLM)
report = report_generator.build_report(sorted_summaries, overview, today)
report_generator.save_report(report, today)                  # 4. 조립 · 저장
```

앞 단계의 결과가 다음 단계의 입력입니다. 중간 결과를 변수에 담아 두어서, 어디서 틀렸는지 찾기 쉽습니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
def make_title(today):
    return f"# 야간 보안 관제 보고 ({today})"

for day in ["2026-10-07", "2026-10-08"]:
    print(make_title(day))
```

막히면 바로 위 `7.1 함수 넷을 차례로 부르면 보고서가 나옵니다` 설명을 다시 봅니다.

In [18]:
def make_title(today):
    return f"# 야간 보안 관제 보고 ({today})"

for day in ["2026-10-07", "2026-10-08"]:
    print(make_title(day))

# 야간 보안 관제 보고 (2026-10-07)
# 야간 보안 관제 보고 (2026-10-08)


✅ 날짜만 다른 제목 두 줄 — 함수는 같고 입력만 바뀌면 모양이 같은 결과가 나옵니다

#### 🐍 문법 상자 · 파일 안의 함수는 `파일이름.함수` 로 부른다

```python
import json
print(json.dumps({"a": 1}))
# {"a": 1}
```

| 쓰는 것 | 뜻 |
|---|---|
| `import report_generator` | `report_generator.py` 를 불러온다 |
| `report_generator.build_report(…)` | 그 파일 안의 `build_report` 함수를 부른다 |
| `from report_generator import build_report` | 이렇게 꺼내면 `build_report(…)` 로 바로 부른다 |

⚠ `json.dumps` 도 같은 모양입니다 — `json.py` 안의 `dumps` 함수입니다. 우리 파일도 똑같이 쓸 수 있습니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-2 · report_generator.py 만들기</font></h3></td></tr></table>

오후의 산출물 **`report_generator.py`** 를 만드시오. `build_report` 의 빈칸 세 곳을 채웁니다.

1. 아래 셀을 채워 실행하면 파일이 만들어집니다.
2. 그 아래 **확인 셀**을 실행합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 문제 셀은 `Writing report_generator.py`, 확인 셀은 `(high 2건)` 이 든 보고서 — 경고 줄 없음 |

**💡 힌트**

1. 빈칸은 6-2 의 세기 · 6-3 의 경고 · `return` 입니다.
2. `make_lines` · `make_overview` · `save_report` 는 5 · 6교시의 줄을 옮겨 둔 것입니다.
3. 확인 셀에서 오류가 나면 문제 셀을 고쳐 다시 실행하고, **확인 셀도 다시** 실행합니다.

In [19]:
%%writefile report_generator.py
from llm_client import call_llm                               # llm_client.py 의 call_llm 을 꺼내 쓴다

# 총평 지시 — 숫자는 코드가 세므로 쓰지 말라고 한다
overview_instruction = "다음은 밤사이 보안 경보 요약입니다. 팀장에게 보고할 총평을 한국어 두세 문장으로 쓰세요. 숫자는 쓰지 마세요. 요약: "


def make_lines(sorted_summaries):                             # 정렬된 요약을 목록 줄로 만드는 함수
    lines = ""                                                # 목록 줄을 모을 빈 문자열
    for s in sorted_summaries:                                # 정렬된 요약을 하나씩 꺼낸다
        # 「- [HIGH] E01 요약」 한 줄을 붙인다
        lines = lines + f"- [{s['risk_level'].upper()}] {s['id']} {s['summary']}\n"
    return lines                                              # 목록 줄을 돌려준다


def make_overview(sorted_summaries):                          # 정렬된 요약으로 총평을 받는 함수
    # 요약 줄을 지시와 함께 보내 총평을 받는다
    text = call_llm(overview_instruction + make_lines(sorted_summaries))
    if text == "":                                            # 답을 받지 못했으면 (한도 초과 등)
        return "총평을 만들지 못했습니다. 건별 내역을 먼저 확인하세요."              # 실패를 숨기지 않고 알린다
    return text.strip()                                       # 앞뒤 공백을 지운 총평을 돌려준다


def build_report(sorted_summaries, overview, today):          # 틀 · 숫자는 코드로, 총평은 받은 문장으로 보고서를 만드는 함수
    total = len(sorted_summaries)                             # 전체 건수 — 코드가 센다
    high_count = 0                                            # high 건수. 0 에서 시작
    # 1. sorted_summaries 를 돌면서 risk_level 을 소문자로 바꾼 값이 "high" 이면 high_count 에 1 을 더하세요


    warning = ""                                              # 기본은 경고 없음
    # 2. high_count 가 3 이상이면 warning 에 f"- **주의: high 경보 {high_count}건 — 건별 내역을 먼저 확인할 것**\n" 을 담으세요


    report = f"# 야간 보안 관제 보고 ({today})\n\n"
    report = report + "## 한눈에 보기\n" + f"- 처리한 경보: {total}건 (high {high_count}건)\n" + warning
    report = report + "\n## 총평\n" + overview + "\n"
    report = report + "\n## 건별 내역 (위험한 것부터)\n" + make_lines(sorted_summaries)
    # 3. report 를 return 하세요



def save_report(report, today):                               # 보고서를 날짜가 든 파일로 저장하는 함수
    filename = f"daily_report_{today.replace('-', '')}.md"    # 날짜의 - 를 빼서 파일 이름을 만든다 — daily_report_20261007.md
    with open(filename, "w", encoding="utf-8") as f:          # 보고서 파일을 쓰기로 연다 (있으면 덮어쓴다)
        f.write(report)                                       # 보고서 문자열을 파일에 쓴다
    return filename                                           # 저장한 파일 이름을 돌려준다

Overwriting report_generator.py


In [20]:
# 확인 셀 — 그대로 실행합니다. report_generator.py 를 고쳤으면 이 셀을 다시 실행합니다
import importlib                                              # 파일을 다시 읽는 도구
import report_generator                                       # 방금 만든 report_generator.py 를 불러온다

importlib.reload(report_generator)                            # 고친 파일을 다시 읽는다

small = [                                                     # high 가 2건뿐인 작은 데이터
    {"id": "E01", "risk_level": "high", "summary": "admin 계정에 로그인 실패 4회"},
    {"id": "E02", "risk_level": "High", "summary": "실패 직후 admin 로그인 성공 — 탈취 의심"},
    {"id": "E05", "risk_level": "medium", "summary": "svc_backup 의 sudoers 접근 거부"},
    {"id": "E06", "risk_level": "low", "summary": "kim01 로그인 실패 1회"},
]

print(report_generator.build_report(small, "총평 문장", "2026-10-08"))

None


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-3 · 오늘 보고서 완성하기</font></h3></td></tr></table>

정렬한 요약 18건으로 총평을 받고, 보고서를 만들어 저장하시오. **LLM 을 한 번 부릅니다.**

| | |
|---|---|
| 🎯 나와야 하는 결과 | `daily_report_20261007.md 를 저장했습니다` 와 보고서 앞부분 — 경고 줄이 있습니다 |

**💡 힌트**

1. `report_generator.` 뒤에 함수 이름을 붙여 부릅니다.
2. 순서는 총평 → 보고서 → 저장입니다.
3. 6교시 6-5 의 파일을 이번에는 진짜 총평으로 덮어씁니다(`"w"`).

In [21]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import importlib                                              # 파일을 다시 읽는 도구
import report_generator                                       # 오후에 만든 report_generator.py 를 불러온다
import event_summarizer                                       # 오전에 만든 event_summarizer.py 를 불러온다

importlib.reload(report_generator)                            # 파일을 고쳤으면 다시 읽는다
importlib.reload(event_summarizer)                            # 파일을 고쳤으면 다시 읽는다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("sorted_summaries.json", encoding="utf-8") as f:   # 준비 셀이 저장한 정렬된 요약
    sorted_summaries = json.load(f)                           # 정렬된 요약 18건을 리스트로 읽는다
today = "2026-10-07"                                          # 보고서 날짜
# ── 여기까지 ──

# 1. overview 에 report_generator.make_overview(sorted_summaries) 의 결과를 담으세요 — LLM 을 한 번 부릅니다
overview = report_generator.make_overview(sorted_summaries)   # 총평을 받는다 — LLM 한 번

# 2. report 에 report_generator.build_report(sorted_summaries, overview, today) 의 결과를 담으세요
# 틀 · 숫자 · 총평으로 보고서를 만든다
report = report_generator.build_report(sorted_summaries, overview, today)

# 3. filename 에 report_generator.save_report(report, today) 의 결과를 담으세요
filename = report_generator.save_report(report, today)        # 날짜가 든 파일로 저장한다

print(filename, "를 저장했습니다")
print(report[0:250])

TypeError: write() argument must be str, not None

#### 🐍 문법 상자 · 함수를 이어 붙이면 파이프라인이 됩니다

```python
def step1(x): return x + 1
def step2(x): return x * 10
print(step2(step1(2)))
# 30
```

| 단계 | 함수 | 파일 | LLM |
|---|---|---|---|
| 1 | `summarize_events(events)` | `event_summarizer.py` | 묶음 수만큼 |
| 2 | `sort_by_risk(summaries)` | `event_summarizer.py` | 안 씀 |
| 3 | `make_overview(sorted_summaries)` | `report_generator.py` | 1번 |
| 4 | `build_report(…)` → `save_report(…)` | `report_generator.py` | 안 씀 |

⚠ 앞 단계의 결과가 다음 단계의 입력입니다. 중간 결과를 변수에 담아 두면 어디서 틀렸는지 찾기 쉽습니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-4 · 다른 날 경보로 처음부터 끝까지</font></h3></td></tr></table>

「내일」 경보 6건으로 요약 → 정렬 → 총평 → 보고서 → 저장을 한 번에 하시오. **한 번 실행에 LLM 을 두 번 부릅니다.**

| | |
|---|---|
| 🎯 나와야 하는 결과 | `daily_report_20261008.md` 와 `경고 줄이 있나: False` — high 가 1건이라 경고가 빠집니다(LLM 판단에 따라 다를 수 있습니다) |

**💡 힌트**

1. 6건은 묶음 하나라서 요약에 LLM 을 한 번 부릅니다.
2. 함수 이름 앞에 파일 이름(`event_summarizer.` · `report_generator.`)을 붙입니다.
3. `"**주의" in report` 는 보고서에 경고 줄이 있는지 봅니다.

In [ ]:
import json                                              # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import importlib                                         # 파일을 다시 읽는 도구
import report_generator                                  # 오후에 만든 report_generator.py 를 불러온다
import event_summarizer                                  # 오전에 만든 event_summarizer.py 를 불러온다

importlib.reload(report_generator)                        # 파일을 고쳤으면 다시 읽는다
importlib.reload(event_summarizer)                        # 파일을 고쳤으면 다시 읽는다

# — 미리 채워 둔 줄입니다. 고치지 않습니다 —
with open("events_1008.json", encoding="utf-8") as f:      # 준비 셀이 저장한 「다른 날」 경보 6건
    events = json.load(f)                                # 경보 목록을 리스트로 읽는다
today = "2026-10-08"                                     # 「다른 날」 보고서 날짜
# — 여기까지 —

# 1. summaries 에 event_summarizer.summarize_events(events) 의 결과를 담으세요 — 6건이라 LLM 을 한 번 부릅니다
summaries = event_summarizer.summarize_events(events)

# 2. sorted_summaries 에 event_summarizer.sort_by_risk(summaries) 의 결과를 담으세요
sorted_summaries = event_summarizer.sort_by_risk(summaries)

# 3. overview 에 report_generator.make_overview(sorted_summaries) 를, report 에 build_report(...) 결과를 담으세요 — LLM 을 한 번 더 부릅니다
overview = report_generator.make_overview(sorted_summaries)
report = report_generator.build_report(sorted_summaries, overview, today)

# 4. report_generator.save_report(report, today) 로 저장하고 파일 이름을 출력하세요
filename = report_generator.save_report(report, today)
print(filename)

print("경고 줄이 있나:", "**주의" in report)

TypeError: can only concatenate str (not "NoneType") to str

#### 🐍 문법 상자 · `in` 으로 글자가 들어 있는지 보기

```python
report = "## 한눈에 보기\n- **주의: high 3건**"
print("**주의" in report, "총평" in report)
# True False
```

| 쓰는 것 | 결과 |
|---|---|
| `"**주의" in report` | 보고서에 그 글자가 있으면 `True` |
| `"x" in ["x", "y"]` | 리스트에 그 값이 있으면 `True` (9/23) |

⚠ 문자열에도 리스트에도 `in` 을 씁니다. 문자열에서는 「글자가 들어 있는가」입니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-5 · 두 보고서의 형식이 같은지 확인하기</font></h3></td></tr></table>

오늘(10/7)과 내일(10/8) 보고서에서 `## ` 로 시작하는 소제목만 뽑아 나란히 출력하시오. LLM 을 부르지 않습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 두 줄 모두 `['## 한눈에 보기', '## 총평', '## 건별 내역 (위험한 것부터)']` |

**💡 힌트**

1. 파일 읽기와 `.startswith` 는 6-5 와 같습니다.
2. 소제목만 고르려면 `"## "` 로 비교합니다.
3. 경보 수 · 총평 문장이 달라도 **소제목은 같아야** 합니다. 학원 기준 「보고서 형식이 매번 같다」입니다.

In [24]:
# — 미리 채워 둔 줄입니다. 고치지 않습니다 —
names = ["daily_report_20261007.md", "daily_report_20261008.md"]
# — 여기까지 —

for name in names:
    headings = []
    
    try:
        # 1. name 을 읽기로 열고 내용을 text 에 담으세요
        with open(name, "r", encoding="utf-8") as f:
            text = f.read()

        # 2. text 를 줄마다 나눠 돌면서, "## " 로 시작하는 줄을 headings 에 append 하세요
        for line in text.splitlines():
            if line.startswith("## "):
                headings.append(line)

        # 3. name 과 headings 를 출력하세요
        print(name, headings)

    except FileNotFoundError:
        print(f"경고: {name} 파일을 찾을 수 없습니다. 이전 문제를 먼저 실행해 주세요.")

daily_report_20261007.md []
경고: daily_report_20261008.md 파일을 찾을 수 없습니다. 이전 문제를 먼저 실행해 주세요.


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>

<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 7-6 · 「사람이 확인할 것」 체크리스트 붙이기</font></h3></td></tr></table>

보고서 맨 아래에 high 경보마다 `- [ ] id 요약` 줄을 붙여, 담당자가 하나씩 확인할 수 있게 하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `## 사람이 확인할 것` 아래 `- [ ] E01 …` · `- [ ] E02 …` 두 줄 |

**💡 힌트**

1. `- [ ]` 는 마크다운 체크 상자입니다. 깃허브에서 보면 네모 칸이 됩니다.
2. high 만 고르는 것은 4-2 · 5-6 과 같습니다.
3. 10/6 의 승인 게이트와 같은 생각입니다 — 판단은 LLM, 확인은 사람.

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
small = [                                                     # high 가 2건뿐인 작은 데이터
    {"id": "E01", "risk_level": "high", "summary": "admin 계정에 로그인 실패 4회"},
    {"id": "E02", "risk_level": "High", "summary": "실패 직후 admin 로그인 성공 — 탈취 의심"},
    {"id": "E05", "risk_level": "medium", "summary": "svc_backup 의 sudoers 접근 거부"},
    {"id": "E06", "risk_level": "low", "summary": "kim01 로그인 실패 1회"},
]
report = "# 야간 보안 관제 보고 (2026-10-08)\n"                 # 보고서의 앞부분이라고 가정합니다
# ── 여기까지 ──

# 1. report 뒤에 "\n## 사람이 확인할 것\n" 을 이어 붙이세요
# 2. small 가운데 risk_level 이 high 인 것마다 "- [ ] id summary" 줄을 이어 붙이세요 (체크 상자 모양)
# 3. report 를 출력하세요


### 📋 7교시 한눈에

| 단계 | 함수 | 파일 |
|---|---|---|
| 묶음 요약 | `summarize_events(events)` | `event_summarizer.py` |
| 위험도순 정렬 | `sort_by_risk(summaries)` | `event_summarizer.py` |
| 총평 | `make_overview(sorted_summaries)` | `report_generator.py` |
| 조립 · 저장 | `build_report(…)` · `save_report(…)` | `report_generator.py` |

오후 산출물은 `agent_core/report_generator.py` 와 `daily_report_20261007.md` 입니다.

---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">8교시 (17:00–17:30) · 오늘 만든 것 확인</mark>

새로 배우는 것은 없습니다. 아래 셀을 실행하면 오늘의 산출물이 있는지 알려 줍니다.

In [ ]:
import os                                                     # 파일이 있는지 보는 도구

# 오늘의 산출물 네 개
for name in ["llm_client.py", "event_summarizer.py", "report_generator.py", "daily_report_20261007.md"]:
    if os.path.exists(name):                                  # 그 파일이 있으면
        print("있음 :", name)
    else:                                                     # 아니면
        print("없음 :", name, "— 그 파일을 만드는 문제로 돌아갑니다")

### 오늘 만든 것

| 파일 | 만든 문제 | 하는 일 |
|---|---|---|
| `event_summarizer.py` | 오전 4-5 | 경보를 묶음으로 요약하고 위험도순으로 정렬한다 |
| `report_generator.py` | 오후 7-2 | 총평을 받고, 틀과 숫자를 코드로 채워 보고서를 만든다 |
| `daily_report_20261007.md` | 오후 7-3 | 오늘의 야간 보안 관제 보고 |

**내일(10/8) 아침 과제**에서 이 파일들을 `git push` 로 깃허브에 올립니다. 오늘은 올리지 않아도 됩니다.

### 남은 도전 문제

| 도전 | 무엇을 해 보나 |
|---|---|
| ⭐5-6 | high 만 골라 「먼저 할 일」 받기 |
| ⭐6-6 | `build_report` 로 묶고 경고 줄이 빠지는지 보기 |
| ⭐7-6 | 「사람이 확인할 것」 체크리스트 붙이기 |

---

# <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">정답 · 먼저 풀어 본 뒤에 봅니다</mark>

출력 예상 문제의 답은 문제 바로 아래 ✅ 줄에 있습니다.

### 정답 5-2

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("sorted_summaries.json", encoding="utf-8") as f:   # 준비 셀이 저장한 정렬된 요약
    sorted_summaries = json.load(f)                           # 정렬된 요약 18건을 리스트로 읽는다
summary_text = ""                                             # LLM 에 보낼 요약 글
# ── 여기까지 ──

# 1. for 로 sorted_summaries 에서 하나씩 꺼내 s 에 담으세요
for s in sorted_summaries:                                    # 정렬된 요약을 하나씩 꺼낸다
    # 2. summary_text 뒤에 f"- [{s['risk_level'].upper()}] {s['summary']}\n" 을 이어 붙이세요
    # 요약 한 줄을 붙인다
    summary_text = summary_text + f"- [{s['risk_level'].upper()}] {s['summary']}\n"

print(summary_text[0:120])                                    # 앞부분만 출력한다
print("…", len(summary_text), "글자")

### 정답 5-3

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import importlib                                              # 파일을 다시 읽는 도구
import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다

importlib.reload(llm_client)                                  # llm_client.py 를 고쳤으면 다시 읽습니다
from llm_client import call_llm                               # 10/6 에 만든 함수를 꺼내 쓴다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("sorted_summaries.json", encoding="utf-8") as f:   # 준비 셀이 저장한 정렬된 요약
    sorted_summaries = json.load(f)                           # 정렬된 요약 18건을 리스트로 읽는다
summary_text = ""                                             # LLM 에 보낼 요약 글
for s in sorted_summaries:                                    # 정렬된 요약을 하나씩 꺼낸다
    # 요약 한 줄을 붙인다
    summary_text = summary_text + f"- [{s['risk_level'].upper()}] {s['summary']}\n"
# 총평 지시 — 숫자는 코드가 세므로 쓰지 말라고 한다
overview_instruction = "다음은 밤사이 보안 경보 요약입니다. 팀장에게 보고할 총평을 한국어 두세 문장으로 쓰세요. 숫자는 쓰지 마세요. 요약: "
# ── 여기까지 ──

# 1. overview 라는 변수에 call_llm(overview_instruction + summary_text) 의 답을 담으세요
overview = call_llm(overview_instruction + summary_text)      # 총평을 받는다 — LLM 한 번

# 2. overview 의 앞뒤 공백을 지운 값을 다시 overview 에 담으세요 (.strip())
overview = overview.strip()                                   # 앞뒤 공백 · 줄바꿈을 지운다

print(overview)                                               # 총평을 본다

### 정답 5-4

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
overview = ""                                                 # LLM 이 답을 못 준 경우라고 가정합니다
# ── 여기까지 ──

# 1. overview 가 빈 문자열이면 (if 문)
if overview == "":                                            # 답을 못 받았으면
    # 2. overview 에 "총평을 만들지 못했습니다. 건별 내역을 먼저 확인하세요." 를 담으세요
    overview = "총평을 만들지 못했습니다. 건별 내역을 먼저 확인하세요."              # 대신 쓸 문장

print(overview)                                               # 총평을 본다

### 정답 5-5

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import importlib                                              # 파일을 다시 읽는 도구
import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다

importlib.reload(llm_client)                                  # llm_client.py 를 고쳤으면 다시 읽습니다
from llm_client import call_llm                               # 10/6 에 만든 함수를 꺼내 쓴다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("sorted_summaries.json", encoding="utf-8") as f:   # 준비 셀이 저장한 정렬된 요약
    sorted_summaries = json.load(f)                           # 정렬된 요약 18건을 리스트로 읽는다
# 총평 지시 — 숫자는 코드가 세므로 쓰지 말라고 한다
overview_instruction = "다음은 밤사이 보안 경보 요약입니다. 팀장에게 보고할 총평을 한국어 두세 문장으로 쓰세요. 숫자는 쓰지 마세요. 요약: "
# ── 여기까지 ──


def make_overview(sorted_summaries):                          # 정렬된 요약으로 총평을 받는 함수
    summary_text = ""                                         # LLM 에 보낼 요약 글
    for s in sorted_summaries:                                # 정렬된 요약을 하나씩 꺼낸다
        # 요약 한 줄을 붙인다
        summary_text = summary_text + f"- [{s['risk_level'].upper()}] {s['summary']}\n"
    # 1. text 라는 변수에 call_llm(overview_instruction + summary_text) 의 답을 담으세요
    text = call_llm(overview_instruction + summary_text)      # 총평을 받는다
    # 2. text 가 빈 문자열이면 "총평을 만들지 못했습니다. 건별 내역을 먼저 확인하세요." 를 return 하세요
    if text == "":                                            # 답을 받지 못했으면 (한도 초과 등)
        return "총평을 만들지 못했습니다. 건별 내역을 먼저 확인하세요."              # 실패를 숨기지 않고 알린다
    # 3. text 의 앞뒤 공백을 지운 값을 return 하세요
    return text.strip()                                       # 앞뒤 공백을 지운 총평을 돌려준다


print(make_overview(sorted_summaries))

### 정답 ⭐5-6

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import importlib                                              # 파일을 다시 읽는 도구
import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다

importlib.reload(llm_client)                                  # llm_client.py 를 고쳤으면 다시 읽습니다
from llm_client import call_llm                               # 10/6 에 만든 함수를 꺼내 쓴다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("sorted_summaries.json", encoding="utf-8") as f:   # 준비 셀이 저장한 정렬된 요약
    sorted_summaries = json.load(f)                           # 정렬된 요약 18건을 리스트로 읽는다
# ── 여기까지 ──

# 1. risk_level 이 high 인 요약만 모아 high_text 라는 문자열로 만드세요 (한 줄에 하나, .lower())
# 2. "다음 high 경보를 보고, 보안 담당자가 가장 먼저 할 조치 한 가지를 한국어 한 문장으로 쓰세요: " 뒤에 high_text 를 붙여 LLM 에 보내세요
# 3. 받은 답을 "먼저 할 일:" 과 함께 출력하세요
high_text = ""                                                # high 요약만 모을 글
for s in sorted_summaries:                                    # 정렬된 요약을 하나씩 꺼낸다
    if s["risk_level"].lower() == "high":                     # 소문자로 맞춰 high 인지 본다
        high_text = high_text + "- " + s["summary"] + "\n"    # high 요약 한 줄을 붙인다

first_action = call_llm("다음 high 경보를 보고, 보안 담당자가 가장 먼저 할 조치 한 가지를 한국어 한 문장으로 쓰세요: " + high_text)
print("먼저 할 일:", first_action.strip())

### 정답 6-2

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("sorted_summaries.json", encoding="utf-8") as f:   # 준비 셀이 저장한 정렬된 요약
    sorted_summaries = json.load(f)                           # 정렬된 요약 18건을 리스트로 읽는다
# ── 여기까지 ──

high_count = 0                                                # high 건수. 0 에서 시작

# 1. total 이라는 변수에 sorted_summaries 의 길이를 담으세요
total = len(sorted_summaries)                                 # 전체 건수 — 코드가 센다

# 2. for 로 sorted_summaries 를 돌면서 risk_level 을 소문자로 바꾼 값이 "high" 이면 high_count 에 1 을 더하세요
for s in sorted_summaries:                                    # 정렬된 요약을 하나씩 꺼낸다
    if s["risk_level"].lower() == "high":                     # 소문자로 맞춰 high 인지 본다
        high_count = high_count + 1                           # high 하나를 센다

# 3. summary_line 에 f"- 처리한 경보: {total}건 (high {high_count}건)\n" 을 담으세요
summary_line = f"- 처리한 경보: {total}건 (high {high_count}건)\n"   # 숫자는 코드가 센 값으로 채운다

print(summary_line)

### 정답 6-3

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
high_count = 6                                                # 문제 6-2 에서 센 값
# ── 여기까지 ──

warning = ""                                                  # 기본은 경고 없음

# 1. high_count 가 3 이상이면 (if 문)
if high_count >= 3:                                           # high 가 3건 이상이면
    # 2. warning 에 f"- **주의: high 경보 {high_count}건 — 건별 내역을 먼저 확인할 것**\n" 을 담으세요
    # 경고 줄 — 보고서 맨 위 「한눈에 보기」에 들어간다
    warning = f"- **주의: high 경보 {high_count}건 — 건별 내역을 먼저 확인할 것**\n"

print(warning)
print("경고 줄 글자 수:", len(warning))

### 정답 6-4

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("sorted_summaries.json", encoding="utf-8") as f:   # 준비 셀이 저장한 정렬된 요약
    sorted_summaries = json.load(f)                           # 정렬된 요약 18건을 리스트로 읽는다
today = "2026-10-07"                                          # 보고서 날짜
summary_line = "- 처리한 경보: 18건 (high 6건)\n"                   # 문제 6-2 의 결과
warning = "- **주의: high 경보 6건 — 건별 내역을 먼저 확인할 것**\n"  # 문제 6-3 의 결과
overview = "밤사이 관리자 계정을 노린 무차별 대입과 로그인 성공이 이어져 계정 탈취가 의심됩니다. 관리자 권한 계정이 새로 만들어진 정황도 있어 우선 확인이 필요합니다."      # 5교시에 LLM 이 쓴 총평이라고 가정합니다
lines = ""                                                    # 건별 내역 줄
for s in sorted_summaries:                                    # 정렬된 요약을 하나씩 꺼낸다
    # 「- [HIGH] E01 요약」 한 줄을 붙인다
    lines = lines + f"- [{s['risk_level'].upper()}] {s['id']} {s['summary']}\n"
# ── 여기까지 ──

# 1. report 에 f"# 야간 보안 관제 보고 ({today})\n\n" 을 담으세요
report = f"# 야간 보안 관제 보고 ({today})\n\n"

# 2. report 뒤에 "## 한눈에 보기\n" + summary_line + warning 을 이어 붙이세요
report = report + "## 한눈에 보기\n" + summary_line + warning

# 3. report 뒤에 "\n## 총평\n" + overview + "\n" 을 이어 붙이세요
report = report + "\n## 총평\n" + overview + "\n"

# 4. report 뒤에 "\n## 건별 내역 (위험한 것부터)\n" + lines 를 이어 붙이세요
report = report + "\n## 건별 내역 (위험한 것부터)\n" + lines

print(report[0:300])

### 정답 6-5

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
today = "2026-10-07"                                          # 보고서 날짜
report = f"# 야간 보안 관제 보고 ({today})\n\n## 한눈에 보기\n- 처리한 경보: 18건 (high 6건)\n"      # 문제 6-4 의 결과를 줄였습니다
report = report + "- **주의: high 경보 6건 — 건별 내역을 먼저 확인할 것**\n\n## 총평\n총평 문장\n\n## 건별 내역 (위험한 것부터)\n- [HIGH] E01 admin 계정에 로그인 실패 4회\n"
# ── 여기까지 ──

# 1. filename 에 f"daily_report_{today.replace('-', '')}.md" 를 담으세요
filename = f"daily_report_{today.replace('-', '')}.md"        # 날짜의 - 를 빼서 파일 이름을 만든다 — daily_report_20261007.md

# 2. filename 을 쓰기로 열고(encoding="utf-8") report 를 저장하세요 (with 문, f.write)
with open(filename, "w", encoding="utf-8") as f:              # 보고서 파일을 쓰기로 연다 (있으면 덮어쓴다)
    f.write(report)                                           # 보고서 문자열을 파일에 쓴다

# 3. filename 을 읽기로 열고 내용을 text 에 담으세요 (f.read())
with open(filename, encoding="utf-8") as f:                   # 저장한 보고서를 읽기로 연다
    text = f.read()                                           # 파일 전체를 문자열 하나로 읽는다

for line in text.split("\n"):                                 # 한 줄씩 나눠
    if line.startswith("#"):                                   # # 으로 시작하는 줄(제목)만
        print(line)

### 정답 ⭐6-6

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
small = [                                                     # high 가 2건뿐인 작은 데이터
    {"id": "E01", "risk_level": "high", "summary": "admin 계정에 로그인 실패 4회"},
    {"id": "E02", "risk_level": "High", "summary": "실패 직후 admin 로그인 성공 — 탈취 의심"},
    {"id": "E05", "risk_level": "medium", "summary": "svc_backup 의 sudoers 접근 거부"},
    {"id": "E06", "risk_level": "low", "summary": "kim01 로그인 실패 1회"},
]
# ── 여기까지 ──

# 1. build_report(sorted_summaries, overview, today) 함수를 만드세요. 문제 6-2 ~ 6-4 의 줄을 함수 안에 모아 report 를 돌려줍니다
# 2. small(high 2건)로 보고서를 만들어 출력하세요. overview 는 "총평 문장", today 는 "2026-10-08" 입니다
# 3. 경고 줄(**주의)이 없는 것을 확인합니다
def build_report(sorted_summaries, overview, today):          # 틀 · 숫자는 코드로, 총평은 받은 문장으로 보고서를 만드는 함수
    total = len(sorted_summaries)                             # 전체 건수 — 코드가 센다
    high_count = 0                                            # high 건수. 0 에서 시작
    for s in sorted_summaries:                                # 정렬된 요약을 하나씩 꺼낸다
        if s["risk_level"].lower() == "high":                 # 소문자로 맞춰 high 인지 본다
            high_count = high_count + 1                       # high 하나를 센다
    warning = ""                                              # 기본은 경고 없음
    if high_count >= 3:                                       # high 가 3건 이상이면
        # 경고 줄 — 보고서 맨 위 「한눈에 보기」에 들어간다
        warning = f"- **주의: high 경보 {high_count}건 — 건별 내역을 먼저 확인할 것**\n"
    lines = ""                                                # 목록 줄을 모을 빈 문자열
    for s in sorted_summaries:                                # 정렬된 요약을 하나씩 꺼낸다
        # 「- [HIGH] E01 요약」 한 줄을 붙인다
        lines = lines + f"- [{s['risk_level'].upper()}] {s['id']} {s['summary']}\n"
    report = f"# 야간 보안 관제 보고 ({today})\n\n"
    report = report + "## 한눈에 보기\n" + f"- 처리한 경보: {total}건 (high {high_count}건)\n" + warning
    report = report + "\n## 총평\n" + overview + "\n"
    report = report + "\n## 건별 내역 (위험한 것부터)\n" + lines
    return report                                             # 완성한 보고서 문자열을 돌려준다


print(build_report(small, "총평 문장", "2026-10-08"))

### 정답 7-2

In [ ]:
%%writefile report_generator.py
from llm_client import call_llm                               # llm_client.py 의 call_llm 을 꺼내 쓴다

# 총평 지시 — 숫자는 코드가 세므로 쓰지 말라고 한다
overview_instruction = "다음은 밤사이 보안 경보 요약입니다. 팀장에게 보고할 총평을 한국어 두세 문장으로 쓰세요. 숫자는 쓰지 마세요. 요약: "


def make_lines(sorted_summaries):                             # 정렬된 요약을 목록 줄로 만드는 함수
    lines = ""                                                # 목록 줄을 모을 빈 문자열
    for s in sorted_summaries:                                # 정렬된 요약을 하나씩 꺼낸다
        # 「- [HIGH] E01 요약」 한 줄을 붙인다
        lines = lines + f"- [{s['risk_level'].upper()}] {s['id']} {s['summary']}\n"
    return lines                                              # 목록 줄을 돌려준다


def make_overview(sorted_summaries):                          # 정렬된 요약으로 총평을 받는 함수
    # 요약 줄을 지시와 함께 보내 총평을 받는다
    text = call_llm(overview_instruction + make_lines(sorted_summaries))
    if text == "":                                            # 답을 받지 못했으면 (한도 초과 등)
        return "총평을 만들지 못했습니다. 건별 내역을 먼저 확인하세요."              # 실패를 숨기지 않고 알린다
    return text.strip()                                       # 앞뒤 공백을 지운 총평을 돌려준다


def build_report(sorted_summaries, overview, today):          # 틀 · 숫자는 코드로, 총평은 받은 문장으로 보고서를 만드는 함수
    total = len(sorted_summaries)                             # 전체 건수 — 코드가 센다
    high_count = 0                                            # high 건수. 0 에서 시작
    # 1. sorted_summaries 를 돌면서 risk_level 을 소문자로 바꾼 값이 "high" 이면 high_count 에 1 을 더하세요
    for s in sorted_summaries:                                # 정렬된 요약을 하나씩 꺼낸다
        if s["risk_level"].lower() == "high":                 # 소문자로 맞춰 high 인지 본다
            high_count = high_count + 1                       # high 하나를 센다

    warning = ""                                              # 기본은 경고 없음
    # 2. high_count 가 3 이상이면 warning 에 f"- **주의: high 경보 {high_count}건 — 건별 내역을 먼저 확인할 것**\n" 을 담으세요
    if high_count >= 3:                                       # high 가 3건 이상이면
        # 경고 줄 — 보고서 맨 위 「한눈에 보기」에 들어간다
        warning = f"- **주의: high 경보 {high_count}건 — 건별 내역을 먼저 확인할 것**\n"

    report = f"# 야간 보안 관제 보고 ({today})\n\n"
    report = report + "## 한눈에 보기\n" + f"- 처리한 경보: {total}건 (high {high_count}건)\n" + warning
    report = report + "\n## 총평\n" + overview + "\n"
    report = report + "\n## 건별 내역 (위험한 것부터)\n" + make_lines(sorted_summaries)
    # 3. report 를 return 하세요
    return report                                             # 완성한 보고서 문자열을 돌려준다


def save_report(report, today):                               # 보고서를 날짜가 든 파일로 저장하는 함수
    filename = f"daily_report_{today.replace('-', '')}.md"    # 날짜의 - 를 빼서 파일 이름을 만든다 — daily_report_20261007.md
    with open(filename, "w", encoding="utf-8") as f:          # 보고서 파일을 쓰기로 연다 (있으면 덮어쓴다)
        f.write(report)                                       # 보고서 문자열을 파일에 쓴다
    return filename                                           # 저장한 파일 이름을 돌려준다

### 정답 7-3

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import importlib                                              # 파일을 다시 읽는 도구
import report_generator                                       # 오후에 만든 report_generator.py 를 불러온다
import event_summarizer                                       # 오전에 만든 event_summarizer.py 를 불러온다

importlib.reload(report_generator)                            # 파일을 고쳤으면 다시 읽는다
importlib.reload(event_summarizer)                            # 파일을 고쳤으면 다시 읽는다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("sorted_summaries.json", encoding="utf-8") as f:   # 준비 셀이 저장한 정렬된 요약
    sorted_summaries = json.load(f)                           # 정렬된 요약 18건을 리스트로 읽는다
today = "2026-10-07"                                          # 보고서 날짜
# ── 여기까지 ──

# 1. overview 에 report_generator.make_overview(sorted_summaries) 의 결과를 담으세요 — LLM 을 한 번 부릅니다
overview = report_generator.make_overview(sorted_summaries)   # 총평을 받는다 — LLM 한 번

# 2. report 에 report_generator.build_report(sorted_summaries, overview, today) 의 결과를 담으세요
# 틀 · 숫자 · 총평으로 보고서를 만든다
report = report_generator.build_report(sorted_summaries, overview, today)

# 3. filename 에 report_generator.save_report(report, today) 의 결과를 담으세요
filename = report_generator.save_report(report, today)        # 날짜가 든 파일로 저장한다

print(filename, "를 저장했습니다")
print(report[0:250])

### 정답 7-4

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import importlib                                              # 파일을 다시 읽는 도구
import report_generator                                       # 오후에 만든 report_generator.py 를 불러온다
import event_summarizer                                       # 오전에 만든 event_summarizer.py 를 불러온다

importlib.reload(report_generator)                            # 파일을 고쳤으면 다시 읽는다
importlib.reload(event_summarizer)                            # 파일을 고쳤으면 다시 읽는다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("events_1008.json", encoding="utf-8") as f:        # 준비 셀이 저장한 「다른 날」 경보 6건
    events = json.load(f)                                     # 경보 목록을 리스트로 읽는다
today = "2026-10-08"                                          # 「다른 날」 보고서 날짜
# ── 여기까지 ──

# 1. summaries 에 event_summarizer.summarize_events(events) 의 결과를 담으세요 — 6건이라 LLM 을 한 번 부릅니다
summaries = event_summarizer.summarize_events(events)         # 묶음으로 나눠 요약한다 — LLM

# 2. sorted_summaries 에 event_summarizer.sort_by_risk(summaries) 의 결과를 담으세요
sorted_summaries = event_summarizer.sort_by_risk(summaries)   # 위험도순으로 정렬한다

# 3. overview 에 report_generator.make_overview(sorted_summaries) 를, report 에 build_report(...) 결과를 담으세요 — LLM 을 한 번 더 부릅니다
overview = report_generator.make_overview(sorted_summaries)   # 총평을 받는다 — LLM 한 번
# 틀 · 숫자 · 총평으로 보고서를 만든다
report = report_generator.build_report(sorted_summaries, overview, today)

# 4. report_generator.save_report(report, today) 로 저장하고 파일 이름을 출력하세요
print(report_generator.save_report(report, today))

print("경고 줄이 있나:", "**주의" in report)

### 정답 7-5

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
names = ["daily_report_20261007.md", "daily_report_20261008.md"]   # 문제 7-3 · 7-4 가 만든 두 보고서
# ── 여기까지 ──

for name in names:                                            # 보고서 파일을 하나씩
    headings = []                                             # 이 보고서의 소제목들
    # 1. name 을 읽기로 열고 내용을 text 에 담으세요
    with open(name, encoding="utf-8") as f:                   # 보고서 파일을 읽기로 연다
        text = f.read()                                       # 파일 전체를 문자열 하나로 읽는다
    # 2. text 를 줄마다 나눠 돌면서, "## " 로 시작하는 줄을 headings 에 append 하세요
    for line in text.split("\n"):                             # 한 줄씩 나눠 본다
        if line.startswith("## "):
            headings.append(line)                             # 소제목 줄을 모은다
    # 3. name 과 headings 를 출력하세요
    print(name, headings)

### 정답 ⭐7-6

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
small = [                                                     # high 가 2건뿐인 작은 데이터
    {"id": "E01", "risk_level": "high", "summary": "admin 계정에 로그인 실패 4회"},
    {"id": "E02", "risk_level": "High", "summary": "실패 직후 admin 로그인 성공 — 탈취 의심"},
    {"id": "E05", "risk_level": "medium", "summary": "svc_backup 의 sudoers 접근 거부"},
    {"id": "E06", "risk_level": "low", "summary": "kim01 로그인 실패 1회"},
]
report = "# 야간 보안 관제 보고 (2026-10-08)\n"                 # 보고서의 앞부분이라고 가정합니다
# ── 여기까지 ──

# 1. report 뒤에 "\n## 사람이 확인할 것\n" 을 이어 붙이세요
# 2. small 가운데 risk_level 이 high 인 것마다 "- [ ] id summary" 줄을 이어 붙이세요 (체크 상자 모양)
# 3. report 를 출력하세요
report = report + "\n## 사람이 확인할 것\n"
for s in small:                                               # 작은 데이터의 요약을 하나씩
    if s["risk_level"].lower() == "high":                     # 소문자로 맞춰 high 인지 본다
        report = report + f"- [ ] {s['id']} {s['summary']}\n"  # 체크 상자 줄을 붙인다

print(report)